# Module 3.5 — Data-Driven Discovery of Unknown Equations (Full Implementation)

Complete implementation of SINDy and symbolic regression for equation discovery.

## Requirements
```
pip install numpy scipy matplotlib scikit-learn pysindy
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sklearn.linear_model import Lasso

## 1. Generate Data from Lorenz System

The Lorenz system:
$$\dot{x} = \sigma(y - x)$$
$$\dot{y} = x(\rho - z) - y$$  
$$\dot{z} = xy - \beta z$$

with $\sigma = 10$, $\rho = 28$, $\beta = 8/3$.

In [ ]:
sigma, rho, beta = 10.0, 28.0, 8.0/3.0

def lorenz(t, state):
    x, y, z = state
    return [sigma*(y - x), x*(rho - z) - y, x*y - beta*z]

# Generate trajectory
dt = 0.001
t_span = (0, 20)
t_eval = np.arange(0, 20, dt)
y0 = [-8, 7, 27]

sol = solve_ivp(lorenz, t_span, y0, t_eval=t_eval, method='RK45', rtol=1e-10)
X_data = sol.y.T  # shape (N, 3)
t_data = sol.t

# Compute derivatives via finite differences (4th order central)
def finite_diff_4th(X, dt):
    dXdt = np.zeros_like(X)
    dXdt[2:-2] = (-X[4:] + 8*X[3:-1] - 8*X[1:-3] + X[:-4]) / (12*dt)
    dXdt[0] = (-25*X[0] + 48*X[1] - 36*X[2] + 16*X[3] - 3*X[4]) / (12*dt)
    dXdt[1] = (-3*X[0] - 10*X[1] + 18*X[2] - 6*X[3] + X[4]) / (12*dt)
    dXdt[-2] = (3*X[-1] + 10*X[-2] - 18*X[-3] + 6*X[-4] - X[-5]) / (12*dt)
    dXdt[-1] = (25*X[-1] - 48*X[-2] + 36*X[-3] - 16*X[-4] + 3*X[-5]) / (12*dt)
    return dXdt

dXdt = finite_diff_4th(X_data, dt)

# Subsample for efficiency
skip = 10
X_sub = X_data[::skip]
dXdt_sub = dXdt[::skip]

print(f'Data shape: {X_sub.shape}')
print(f'Derivative shape: {dXdt_sub.shape}')

# 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot(X_data[:, 0], X_data[:, 1], X_data[:, 2], lw=0.3)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Lorenz Attractor')
plt.show()

## 2. Build SINDy Library

The library $\Theta(X)$ contains candidate nonlinear functions:
$$\Theta = [1, x, y, z, x^2, xy, xz, y^2, yz, z^2, \ldots]$$

We seek sparse coefficient vectors $\Xi$ such that $\dot{X} = \Theta(X)\Xi$.

In [ ]:
def build_library(X, poly_order=2, include_sine=False):
    """Build the SINDy library matrix."""
    n_samples, n_vars = X.shape
    lib = [np.ones(n_samples)]  # Constant term
    labels = ['1']
    var_names = ['x', 'y', 'z'][:n_vars]
    
    # Linear terms
    for i, name in enumerate(var_names):
        lib.append(X[:, i])
        labels.append(name)
    
    # Quadratic terms
    if poly_order >= 2:
        for i in range(n_vars):
            for j in range(i, n_vars):
                lib.append(X[:, i] * X[:, j])
                if i == j:
                    labels.append(f'{var_names[i]}^2')
                else:
                    labels.append(f'{var_names[i]}{var_names[j]}')
    
    # Cubic terms
    if poly_order >= 3:
        for i in range(n_vars):
            for j in range(i, n_vars):
                for k in range(j, n_vars):
                    lib.append(X[:, i] * X[:, j] * X[:, k])
                    labels.append(f'{var_names[i]}{var_names[j]}{var_names[k]}')
    
    if include_sine:
        for i, name in enumerate(var_names):
            lib.append(np.sin(X[:, i]))
            labels.append(f'sin({name})')
    
    return np.column_stack(lib), labels

Theta, labels = build_library(X_sub, poly_order=2)
print(f'Library shape: {Theta.shape}')
print(f'Library terms: {labels}')

## 3. Sequential Thresholded Least Squares (STLS)

In [ ]:
def sindy_stls(Theta, dXdt, threshold=0.1, max_iter=20):
    """SINDy with sequential thresholded least squares."""
    n_targets = dXdt.shape[1]
    n_features = Theta.shape[1]
    
    # Initial least squares
    Xi = np.linalg.lstsq(Theta, dXdt, rcond=None)[0]
    
    for iteration in range(max_iter):
        # Threshold small coefficients
        small_idx = np.abs(Xi) < threshold
        Xi[small_idx] = 0
        
        # Re-solve for remaining coefficients
        for j in range(n_targets):
            big_idx = ~small_idx[:, j]
            if np.sum(big_idx) > 0:
                Xi[big_idx, j] = np.linalg.lstsq(
                    Theta[:, big_idx], dXdt[:, j], rcond=None
                )[0]
    
    return Xi

Xi = sindy_stls(Theta, dXdt_sub, threshold=0.5)

# Print discovered equations
var_names = ['x', 'y', 'z']
print('Discovered Equations:')
print('=' * 50)
for j in range(3):
    terms = []
    for i, (coeff, label) in enumerate(zip(Xi[:, j], labels)):
        if abs(coeff) > 1e-6:
            terms.append(f'{coeff:+.2f}*{label}')
    equation = ' '.join(terms) if terms else '0'
    print(f'd{var_names[j]}/dt = {equation}')

print('\nTrue Equations:')
print(f'dx/dt = {sigma:.1f}*(y - x) = -{sigma:.1f}*x + {sigma:.1f}*y')
print(f'dy/dt = {rho:.1f}*x - y - x*z')
print(f'dz/dt = x*y - {beta:.4f}*z')

## 4. Validation: Simulate Discovered System

In [ ]:
def discovered_dynamics(t, state):
    X_point = np.array(state).reshape(1, -1)
    Theta_point, _ = build_library(X_point, poly_order=2)
    dxdt = Theta_point @ Xi
    return dxdt.flatten()

# Simulate discovered system
sol_discovered = solve_ivp(discovered_dynamics, (0, 20), y0, 
                           t_eval=t_eval, method='RK45', rtol=1e-8)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for i, name in enumerate(['x', 'y', 'z']):
    axes[i].plot(t_data, X_data[:, i], 'b-', lw=0.5, label='True')
    axes[i].plot(sol_discovered.t, sol_discovered.y[i], 'r--', lw=0.5, label='SINDy')
    axes[i].set_ylabel(name)
    axes[i].legend(loc='upper right')
axes[-1].set_xlabel('Time')
axes[0].set_title('True vs SINDy-Discovered Lorenz System')
plt.tight_layout()
plt.show()

## 5. Noise Robustness Analysis

In [ ]:
noise_levels = [0, 0.01, 0.05, 0.1, 0.2, 0.5]
results = []

for noise in noise_levels:
    X_noisy = X_sub + noise * np.std(X_sub, axis=0) * np.random.randn(*X_sub.shape)
    dXdt_noisy = finite_diff_4th(X_data + noise * np.std(X_data, axis=0) * np.random.randn(*X_data.shape), dt)[::skip]
    Theta_noisy, _ = build_library(X_noisy, poly_order=2)
    Xi_noisy = sindy_stls(Theta_noisy, dXdt_noisy, threshold=0.5)
    
    # Count nonzero terms (should be 7 for Lorenz)
    n_nonzero = np.sum(np.abs(Xi_noisy) > 1e-6)
    coeff_error = np.linalg.norm(Xi_noisy - Xi) / max(np.linalg.norm(Xi), 1e-10)
    results.append((noise, n_nonzero, coeff_error))
    print(f'Noise={noise:.2f}: {n_nonzero} nonzero terms, relative error={coeff_error:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot([r[0] for r in results], [r[1] for r in results], 'bo-')
ax1.axhline(y=7, color='r', ls='--', label='True (7 terms)')
ax1.set_xlabel('Noise level'); ax1.set_ylabel('Nonzero terms')
ax1.legend(); ax1.set_title('Sparsity vs Noise')

ax2.plot([r[0] for r in results], [r[2] for r in results], 'ro-')
ax2.set_xlabel('Noise level'); ax2.set_ylabel('Relative coefficient error')
ax2.set_title('Accuracy vs Noise')
plt.tight_layout()
plt.show()